# Transfer Learning on YOLOv11n — Feature Extraction + Fine-Tuning

## Goal
Apply two-stage transfer learning to YOLOv11n on NEU-DET, using the **exact same training recipe** as `week4_fresh_baseline`.

## Two Stages
| Stage | What | Frozen | Trainable | LR | Epochs |
|-------|------|--------|-----------|-----|--------|
| **1: Feature Extraction** | Load COCO → freeze all → train head | 0-22 | 23 (head) | 0.001 | 100 |
| **2: Fine-Tuning** | Load Stage 1 → unfreeze top → low LR | 0-7 | 8-23 (neck+head) | 0.0001 | 200 |

## Model: YOLOv11n (24 layers, 2.62M params)
```
BACKBONE (Layers 0-10): 149 leaf layers — Low-level (edges, textures)
NECK     (Layers 11-22): 84 leaf layers — Mid-level (multi-scale fusion)
HEAD     (Layer 23):     77 leaf layers — High-level (classification, bbox)
```

## Evaluation
- All results on **TEST set** (not train or val)
- Train/val loss plots for overfitting detection
- Comparison: Stage 1 vs Stage 2 vs Baseline (78.8%)

In [ ]:
# ============================================================
# Cell 2: Setup — Paths, Seeds, Device
# ============================================================
import torch
import ultralytics
from pathlib import Path
import json
import sys
import random
import numpy as np

# ── Reproducibility ──
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Paths ──
ROOT = Path(r"D:/DigiSteel-Yolo/DigiSteel-YOLO")
DATA_YAML = ROOT / "configs" / "data" / "neu_det.yaml"
RUNS_DIR = ROOT / "runs" / "detect"
EVALS_DIR = ROOT / "evals"
EVALS_DIR.mkdir(parents=True, exist_ok=True)

# Stage 1 paths
S1_RUN_NAME = "tl_stage1_feature_extraction"
S1_BEST_PT = RUNS_DIR / S1_RUN_NAME / "weights" / "best.pt"
S1_RESULTS_CSV = RUNS_DIR / S1_RUN_NAME / "results.csv"

# Stage 2 paths
S2_RUN_NAME = "tl_stage2_fine_tuning"
S2_BEST_PT = RUNS_DIR / S2_RUN_NAME / "weights" / "best.pt"
S2_RESULTS_CSV = RUNS_DIR / S2_RUN_NAME / "results.csv"

# ── Verify ──
print(f"{'='*60}")
print("  TRANSFER LEARNING SETUP")
print(f"{'='*60}")
print(f"  Device:       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA:         {torch.cuda.is_available()}")
print(f"  Ultralytics:  {ultralytics.__version__}")
print(f"  Data YAML:    {DATA_YAML} (exists: {DATA_YAML.exists()})")
print(f"  Stage 1 out:  {S1_BEST_PT}")
print(f"  Stage 2 out:  {S2_BEST_PT}")

# Verify dataset
import yaml
with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)
base_path = ROOT / data_cfg.get('path', '')
for split in ['train', 'val', 'test']:
    split_path = base_path / data_cfg.get(split, '')
    if split_path.exists():
        count = len(list(split_path.glob('*.jpg'))) + len(list(split_path.glob('*.png')))
        print(f"  {split}: {count} images")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 3: Load Model + Layer Inspection
# ============================================================
from ultralytics import YOLO

print("=" * 70)
print("  YOLOv11n — LAYER INSPECTION")
print("=" * 70)

# Load model (architecture + weights in one step)
model = YOLO('yolo11n.pt')

# Print summary
print("\n[1] Model Summary:\n")
model.info(verbose=True)

# Layer table
print("\n[2] Layer Table:\n")
print(f"{'Idx':<5} {'Type':<20} {'Params':>12} {'Sub-layers':>12} {'Category'}")
print("-" * 70)

layers = list(model.model.model.children())
backbone_p = neck_p = head_p = 0
backbone_l = neck_l = head_l = 0

for i, layer in enumerate(layers):
    ltype = type(layer).__name__
    nparams = sum(p.numel() for p in layer.parameters())
    nsub = sum(1 for _ in layer.modules()) - 1

    if ltype == 'Detect':
        cat = 'HEAD (high-level)'
        head_p += nparams; head_l += nsub
    elif ltype in ('Upsample', 'Concat'):
        cat = 'NECK (mid-level)'
        neck_p += nparams; neck_l += nsub
    elif i >= 11:
        cat = 'NECK (mid-level)'
        neck_p += nparams; neck_l += nsub
    else:
        cat = 'BACKBONE (low-level)'
        backbone_p += nparams; backbone_l += nsub

    print(f"{i:<5} {ltype:<20} {nparams:>12,} {nsub:>12,}   {cat}")

print("-" * 70)
total_l = backbone_l + neck_l + head_l
total_p = backbone_p + neck_p + head_p

print(f"\n{'Section':<30} {'Layers':>8} {'%':>8} {'Params':>12} {'% Params':>8}")
print("-" * 70)
print(f"{'BACKBONE (low-level)':<30} {backbone_l:>8} {backbone_l/total_l*100:>7.1f}% {backbone_p:>12,} {backbone_p/total_p*100:>7.1f}%")
print(f"{'NECK (mid-level)':<30} {neck_l:>8} {neck_l/total_l*100:>7.1f}% {neck_p:>12,} {neck_p/total_p*100:>7.1f}%")
print(f"{'HEAD (high-level)':<30} {head_l:>8} {head_l/total_l*100:>7.1f}% {head_p:>12,} {head_p/total_p*100:>7.1f}%")
print("-" * 70)
print(f"{'TOTAL':<30} {total_l:>8} {'100.0%':>8} {total_p:>12,} {'100.0%':>8}")

print("\n" + "=" * 70)

---
## Stage 1: Feature Extraction

**Strategy:** Freeze ALL backbone + neck (layers 0-22), train ONLY detection head (layer 23)

**Why:** COCO pretrained features (edges, textures) are universal. Only the head needs to learn: "which pattern = which defect class"

**LR:** 0.001 (standard — head needs to learn fast)

In [ ]:
# ============================================================
# Cell 4: Stage 1 — Freeze All Layers
# ============================================================
from ultralytics import YOLO

print("=" * 70)
print("  STAGE 1: FREEZE ALL LAYERS")
print("  Freeze: 0-22 (backbone + neck) | Train: 23 (head)")
print("=" * 70)

# Load fresh model
s1_model = YOLO('yolo11n.pt')

# Freeze all except detection head
frozen_params = 0
trainable_params = 0

for name, param in s1_model.model.named_parameters():
    if 'model.23' in name or 'detect' in name.lower():
        param.requires_grad = True
        trainable_params += param.numel()
    else:
        param.requires_grad = False
        frozen_params += param.numel()

print(f"\n  Frozen:     {frozen_params:>12,} params")
print(f"  Trainable:  {trainable_params:>12,} params")
print(f"  Frozen %:   {frozen_params/(frozen_params+trainable_params)*100:.1f}%")

# Verify layer status
print(f"\n{'Idx':<5} {'Type':<20} {'Params':>12} {'Status':>12}")
print("-" * 55)
for i, layer in enumerate(s1_model.model.model.children()):
    ltype = type(layer).__name__
    nparams = sum(p.numel() for p in layer.parameters())
    is_frozen = all(not p.requires_grad for p in layer.parameters()) if list(layer.parameters()) else True
    status = "LOCK FROZEN" if is_frozen else "FIRE TRAINABLE"
    print(f"{i:<5} {ltype:<20} {nparams:>12,} {status:>12}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 5: Stage 1 — Train (Feature Extraction)
# ============================================================
from ultralytics import YOLO
import time

print("=" * 70)
print("  STAGE 1: TRAINING (Feature Extraction)")
print("  Frozen: ALL (0-22) | Trainable: HEAD (23)")
print("=" * 70)

# Reload fresh model (cell-independent)
s1_model = YOLO('yolo11n.pt')

# Freeze all except head
for name, param in s1_model.model.named_parameters():
    if 'model.23' in name or 'detect' in name.lower():
        param.requires_grad = True
    else:
        param.requires_grad = False

# Training recipe (matching fresh_baseline)
s1_overrides = {
    'data': str(DATA_YAML),
    'task': 'detect',
    'epochs': 100,
    'patience': 30,
    'batch': 16,
    'imgsz': 800,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'mosaic': 0.0,
    'mixup': 0.15,
    'degrees': 10.0,
    'translate': 0.2,
    'scale': 0.6,
    'shear': 5.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'erasing': 0.4,
    'cos_lr': True,
    'deterministic': True,
    'close_mosaic': 10,
    'amp': True,
    'seed': 42,
    'workers': 8,
    'save': True,
    'save_period': 25,
    'plots': True,
    'project': str(RUNS_DIR),
    'name': S1_RUN_NAME,
    'exist_ok': True,
    'verbose': True,
}

print(f"\n  Epochs:    {s1_overrides['epochs']}")
print(f"  LR:        {s1_overrides['lr0']} (cosine decay to {s1_overrides['lrf']})")
print(f"  Patience:  {s1_overrides['patience']}")
print(f"  Batch:     {s1_overrides['batch']}")
print(f"  imgsz:     {s1_overrides['imgsz']}")
print(f"  Mosaic:    {s1_overrides['mosaic']} (disabled)")

# Train
s1_start = time.time()
s1_results = s1_model.train(**s1_overrides)
s1_time = time.time() - s1_start

print(f"\n{'='*60}")
print(f"  STAGE 1 TRAINING COMPLETE")
print(f"  Duration: {s1_time/3600:.1f} hours")
print(f"  Weights:  {S1_BEST_PT}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 6: Stage 1 — Evaluate on TEST Set
# ============================================================
from ultralytics import YOLO
import json

print("=" * 70)
print("  STAGE 1: TEST SET EVALUATION")
print("=" * 70)

if not S1_BEST_PT.exists():
    print(f"  ERROR: Weights not found: {S1_BEST_PT}")
    print("  Run Cell 5 first.")
else:
    s1_best = YOLO(str(S1_BEST_PT))

    # Evaluate on TEST set
    print(f"\n  Evaluating on TEST set...")
    s1_metrics = s1_best.val(
        data=str(DATA_YAML),
        split='test',
        imgsz=800,
        batch=16,
        device=0,
        plots=True,
        verbose=True,
    )

    # Extract metrics
    s1_map50 = float(s1_metrics.box.map50)
    s1_map5095 = float(s1_metrics.box.map)
    s1_prec = float(s1_metrics.box.mp)
    s1_rec = float(s1_metrics.box.mr)

    # Per-class
    class_names = s1_best.names
    s1_per_class = {}
    for cid, cname in class_names.items():
        if cid < len(s1_metrics.box.ap50):
            s1_per_class[cname] = float(s1_metrics.box.ap50[cid])

    print(f"\n{'='*60}")
    print("STAGE 1 TEST RESULTS")
    print(f"{'='*60}")
    print(f"  mAP@0.5:      {s1_map50:.4f} ({s1_map50*100:.1f}%)")
    print(f"  mAP@0.5:0.95: {s1_map5095:.4f} ({s1_map5095*100:.1f}%)")
    print(f"  Precision:    {s1_prec:.4f} ({s1_prec*100:.1f}%)")
    print(f"  Recall:       {s1_rec:.4f} ({s1_rec*100:.1f}%)")
    print(f"\n  Per-class AP@0.5:")
    for cname in sorted(s1_per_class):
        bar = '█' * int(s1_per_class[cname] * 40)
        print(f"    {cname:<18} {s1_per_class[cname]*100:.1f}% {bar}")

    # Save
    s1_eval = {
        "stage": "feature_extraction",
        "mAP50": s1_map50,
        "mAP50_95": s1_map5095,
        "precision": s1_prec,
        "recall": s1_rec,
        "per_class": s1_per_class,
    }
    with open(EVALS_DIR / "tl_stage1_results.json", "w") as f:
        json.dump(s1_eval, f, indent=2)
    print(f"\n  Saved: {EVALS_DIR / 'tl_stage1_results.json'}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 7: Stage 1 — Loss Plots + Overfitting Analysis
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("=" * 70)
print("  STAGE 1: LOSS ANALYSIS & OVERFITTING DETECTION")
print("=" * 70)

if not S1_RESULTS_CSV.exists():
    print(f"  ERROR: results.csv not found: {S1_RESULTS_CSV}")
else:
    df = pd.read_csv(S1_RESULTS_CSV)
    df.columns = df.columns.str.strip()
    print(f"  Loaded: {len(df)} epochs")

    # Find loss columns
    train_loss_cols = [c for c in df.columns if 'train' in c.lower() and 'loss' in c.lower()]
    val_loss_cols = [c for c in df.columns if 'val' in c.lower() and 'loss' in c.lower()]
    all_loss_cols = [c for c in df.columns if 'loss' in c.lower()]

    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Stage 1: Feature Extraction — Training Analysis', fontsize=14, fontweight='bold')

    # Plot 1: All loss curves
    ax = axes[0, 0]
    for col in all_loss_cols:
        ax.plot(df[col], label=col, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('All Loss Curves')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Plot 2: Train vs Val loss (total)
    ax = axes[0, 1]
    train_total = [c for c in df.columns if 'train/box_loss' in c.lower() or c.lower() == 'train/loss']
    val_total = [c for c in df.columns if 'val/box_loss' in c.lower() or c.lower() == 'val/loss']
    if train_total:
        ax.plot(df[train_total[0]], label='Train Loss', color='blue', linewidth=2)
    if val_total:
        ax.plot(df[val_total[0]], label='Val Loss', color='red', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Train vs Val Loss (Overfitting Check)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Add overfitting annotation
    if train_total and val_total:
        train_vals = df[train_total[0]].values
        val_vals = df[val_total[0]].values
        gap = val_vals - train_vals
        final_gap = gap[-1]
        if final_gap > 0.5:
            ax.annotate(f'OVERFITTING\nGap={final_gap:.2f}', xy=(len(gap)-1, val_vals[-1]),
                       fontsize=10, color='red', fontweight='bold')
        else:
            ax.annotate(f'OK\nGap={final_gap:.2f}', xy=(len(gap)-1, val_vals[-1]),
                       fontsize=10, color='green', fontweight='bold')

    # Plot 3: mAP curves
    ax = axes[1, 0]
    map50_col = [c for c in df.columns if 'map50' in c.lower() and 'map50-' not in c.lower()]
    map_col = [c for c in df.columns if 'map50-95' in c.lower()]
    if map50_col:
        ax.plot(df[map50_col[0]], label='mAP@0.5', color='blue', linewidth=2)
    if map_col:
        ax.plot(df[map_col[0]], label='mAP@0.5:0.95', color='red', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP')
    ax.set_title('mAP Progress')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Plot 4: Train-Val Gap (overfitting indicator)
    ax = axes[1, 1]
    if train_total and val_total:
        gap = df[val_total[0]].values - df[train_total[0]].values
        ax.plot(gap, color='purple', linewidth=2)
        ax.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Perfect fit')
        ax.fill_between(range(len(gap)), 0, gap, alpha=0.3,
                       color='red' if gap[-1] > 0 else 'green')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Val - Train Loss')
        ax.set_title('Overfitting Gap (Val - Train)')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Diagnosis
        if gap[-1] > 1.0:
            diagnosis = "OVERFITTING: Val loss >> Train loss"
        elif gap[-1] > 0.5:
            diagnosis = "MILD OVERFITTING: Gap is growing"
        elif gap[-1] < -0.5:
            diagnosis = "UNDERFITTING: Train loss >> Val loss"
        else:
            diagnosis = "GOOD FIT: Gap is small and stable"
        ax.set_xlabel(f'Epoch — {diagnosis}')

    plt.tight_layout()
    s1_plots_path = EVALS_DIR / 'tl_stage1_loss_analysis.png'
    plt.savefig(s1_plots_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n  Saved: {s1_plots_path}")

print("\n" + "=" * 70)

---
## Stage 2: Fine-Tuning

**Strategy:** Freeze early backbone (0-7), unfreeze late backbone + neck + head (8-23)

**Why:** Early layers learn universal features (edges, textures). Late layers need to adapt to steel-specific patterns.

**LR:** 0.0001 constant (very conservative — don't destroy pretrained weights)

In [ ]:
# ============================================================
# Cell 8: Stage 2 — Unfreeze Top Layers
# ============================================================
from ultralytics import YOLO

print("=" * 70)
print("  STAGE 2: UNFREEZE TOP LAYERS")
print("  Freeze: 0-7 (early backbone) | Train: 8-23 (late + neck + head)")
print("=" * 70)

# Load Stage 1 best weights
if not S1_BEST_PT.exists():
    print(f"  ERROR: Stage 1 weights not found: {S1_BEST_PT}")
    print("  Run Cell 5 first.")
else:
    s2_model = YOLO(str(S1_BEST_PT))
    print(f"  Loaded Stage 1 weights: {S1_BEST_PT}")

    # Freeze boundary
    FREEZE_UP_TO = 7  # Freeze layers 0-7

    frozen_params = 0
    trainable_params = 0

    for name, param in s2_model.model.named_parameters():
        parts = name.split('.')
        if parts[0] == 'model' and len(parts) > 1:
            try:
                layer_idx = int(parts[1])
            except ValueError:
                layer_idx = -1

            if layer_idx <= FREEZE_UP_TO:
                param.requires_grad = False
                frozen_params += param.numel()
            else:
                param.requires_grad = True
                trainable_params += param.numel()
        else:
            param.requires_grad = True
            trainable_params += param.numel()

    print(f"\n  Frozen:     {frozen_params:>12,} params (layers 0-{FREEZE_UP_TO})")
    print(f"  Trainable:  {trainable_params:>12,} params (layers {FREEZE_UP_TO+1}-23)")
    print(f"  Frozen %:   {frozen_params/(frozen_params+trainable_params)*100:.1f}%")

    # Verify layer status
    print(f"\n{'Idx':<5} {'Type':<20} {'Params':>12} {'Status':>12}")
    print("-" * 55)
    for i, layer in enumerate(s2_model.model.model.children()):
        ltype = type(layer).__name__
        nparams = sum(p.numel() for p in layer.parameters())
        is_frozen = all(not p.requires_grad for p in layer.parameters()) if list(layer.parameters()) else True
        status = "LOCK FROZEN" if is_frozen else "FIRE TRAINABLE"
        print(f"{i:<5} {ltype:<20} {nparams:>12,} {status:>12}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 9: Stage 2 — Train (Fine-Tuning with Low LR)
# ============================================================
from ultralytics import YOLO
import time

print("=" * 70)
print("  STAGE 2: TRAINING (Fine-Tuning)")
print("  Frozen: 0-7 | Trainable: 8-23 | LR: 0.0001 constant")
print("=" * 70)

# Reload Stage 1 weights (cell-independent)
s2_model = YOLO(str(S1_BEST_PT))

# Apply freezing
FREEZE_UP_TO = 7
for name, param in s2_model.model.named_parameters():
    parts = name.split('.')
    if parts[0] == 'model' and len(parts) > 1:
        try:
            layer_idx = int(parts[1])
        except ValueError:
            layer_idx = -1
        if layer_idx <= FREEZE_UP_TO:
            param.requires_grad = False
        else:
            param.requires_grad = True
    else:
        param.requires_grad = True

# Training recipe (low LR, constant)
s2_overrides = {
    'data': str(DATA_YAML),
    'task': 'detect',
    'epochs': 200,
    'patience': 50,
    'batch': 16,
    'imgsz': 800,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 0.0001,          # LOW LR — conservative
    'lrf': 0.0001,          # Constant — no decay
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'mosaic': 0.0,
    'mixup': 0.15,
    'degrees': 10.0,
    'translate': 0.2,
    'scale': 0.6,
    'shear': 5.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'erasing': 0.4,
    'cos_lr': False,         # No cosine — constant LR
    'deterministic': True,
    'close_mosaic': 10,
    'amp': True,
    'seed': 42,
    'workers': 8,
    'save': True,
    'save_period': 25,
    'plots': True,
    'project': str(RUNS_DIR),
    'name': S2_RUN_NAME,
    'exist_ok': True,
    'verbose': True,
}

print(f"\n  Epochs:    {s2_overrides['epochs']}")
print(f"  LR:        {s2_overrides['lr0']} (constant — no decay)")
print(f"  Patience:  {s2_overrides['patience']}")
print(f"  Batch:     {s2_overrides['batch']}")
print(f"  imgsz:     {s2_overrides['imgsz']}")

# Train
s2_start = time.time()
s2_results = s2_model.train(**s2_overrides)
s2_time = time.time() - s2_start

print(f"\n{'='*60}")
print(f"  STAGE 2 TRAINING COMPLETE")
print(f"  Duration: {s2_time/3600:.1f} hours")
print(f"  Weights:  {S2_BEST_PT}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 10: Stage 2 — Evaluate on TEST Set
# ============================================================
from ultralytics import YOLO
import json

print("=" * 70)
print("  STAGE 2: TEST SET EVALUATION")
print("=" * 70)

if not S2_BEST_PT.exists():
    print(f"  ERROR: Weights not found: {S2_BEST_PT}")
    print("  Run Cell 9 first.")
else:
    s2_best = YOLO(str(S2_BEST_PT))

    print(f"\n  Evaluating on TEST set...")
    s2_metrics = s2_best.val(
        data=str(DATA_YAML),
        split='test',
        imgsz=800,
        batch=16,
        device=0,
        plots=True,
        verbose=True,
    )

    s2_map50 = float(s2_metrics.box.map50)
    s2_map5095 = float(s2_metrics.box.map)
    s2_prec = float(s2_metrics.box.mp)
    s2_rec = float(s2_metrics.box.mr)

    class_names = s2_best.names
    s2_per_class = {}
    for cid, cname in class_names.items():
        if cid < len(s2_metrics.box.ap50):
            s2_per_class[cname] = float(s2_metrics.box.ap50[cid])

    print(f"\n{'='*60}")
    print("STAGE 2 TEST RESULTS")
    print(f"{'='*60}")
    print(f"  mAP@0.5:      {s2_map50:.4f} ({s2_map50*100:.1f}%)")
    print(f"  mAP@0.5:0.95: {s2_map5095:.4f} ({s2_map5095*100:.1f}%)")
    print(f"  Precision:    {s2_prec:.4f} ({s2_prec*100:.1f}%)")
    print(f"  Recall:       {s2_rec:.4f} ({s2_rec*100:.1f}%)")
    print(f"\n  Per-class AP@0.5:")
    for cname in sorted(s2_per_class):
        bar = '█' * int(s2_per_class[cname] * 40)
        print(f"    {cname:<18} {s2_per_class[cname]*100:.1f}% {bar}")

    s2_eval = {
        "stage": "fine_tuning",
        "mAP50": s2_map50,
        "mAP50_95": s2_map5095,
        "precision": s2_prec,
        "recall": s2_rec,
        "per_class": s2_per_class,
    }
    with open(EVALS_DIR / "tl_stage2_results.json", "w") as f:
        json.dump(s2_eval, f, indent=2)
    print(f"\n  Saved: {EVALS_DIR / 'tl_stage2_results.json'}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 11: Stage 2 — Loss Plots + Overfitting Analysis
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd

print("=" * 70)
print("  STAGE 2: LOSS ANALYSIS & OVERFITTING DETECTION")
print("=" * 70)

if not S2_RESULTS_CSV.exists():
    print(f"  ERROR: results.csv not found: {S2_RESULTS_CSV}")
else:
    df = pd.read_csv(S2_RESULTS_CSV)
    df.columns = df.columns.str.strip()
    print(f"  Loaded: {len(df)} epochs")

    all_loss_cols = [c for c in df.columns if 'loss' in c.lower()]
    train_total = [c for c in df.columns if 'train/box_loss' in c.lower() or c.lower() == 'train/loss']
    val_total = [c for c in df.columns if 'val/box_loss' in c.lower() or c.lower() == 'val/loss']

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Stage 2: Fine-Tuning — Training Analysis', fontsize=14, fontweight='bold')

    # Plot 1: All loss curves
    ax = axes[0, 0]
    for col in all_loss_cols:
        ax.plot(df[col], label=col, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('All Loss Curves')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Plot 2: Train vs Val loss
    ax = axes[0, 1]
    if train_total:
        ax.plot(df[train_total[0]], label='Train Loss', color='blue', linewidth=2)
    if val_total:
        ax.plot(df[val_total[0]], label='Val Loss', color='red', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Train vs Val Loss (Overfitting Check)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    if train_total and val_total:
        gap = df[val_total[0]].values - df[train_total[0]].values
        final_gap = gap[-1]
        if final_gap > 0.5:
            ax.annotate(f'OVERFITTING\nGap={final_gap:.2f}', xy=(len(gap)-1, df[val_total[0]].values[-1]),
                       fontsize=10, color='red', fontweight='bold')
        else:
            ax.annotate(f'OK\nGap={final_gap:.2f}', xy=(len(gap)-1, df[val_total[0]].values[-1]),
                       fontsize=10, color='green', fontweight='bold')

    # Plot 3: mAP curves
    ax = axes[1, 0]
    map50_col = [c for c in df.columns if 'map50' in c.lower() and 'map50-' not in c.lower()]
    map_col = [c for c in df.columns if 'map50-95' in c.lower()]
    if map50_col:
        ax.plot(df[map50_col[0]], label='mAP@0.5', color='blue', linewidth=2)
    if map_col:
        ax.plot(df[map_col[0]], label='mAP@0.5:0.95', color='red', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP')
    ax.set_title('mAP Progress')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Plot 4: Overfitting Gap
    ax = axes[1, 1]
    if train_total and val_total:
        gap = df[val_total[0]].values - df[train_total[0]].values
        ax.plot(gap, color='purple', linewidth=2)
        ax.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Perfect fit')
        ax.fill_between(range(len(gap)), 0, gap, alpha=0.3,
                       color='red' if gap[-1] > 0 else 'green')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Val - Train Loss')
        ax.set_title('Overfitting Gap (Val - Train)')
        ax.legend()
        ax.grid(True, alpha=0.3)

        if gap[-1] > 1.0:
            diagnosis = "OVERFITTING: Val loss >> Train loss"
        elif gap[-1] > 0.5:
            diagnosis = "MILD OVERFITTING: Gap is growing"
        elif gap[-1] < -0.5:
            diagnosis = "UNDERFITTING: Train loss >> Val loss"
        else:
            diagnosis = "GOOD FIT: Gap is small and stable"
        ax.set_xlabel(f'Epoch — {diagnosis}')

    plt.tight_layout()
    s2_plots_path = EVALS_DIR / 'tl_stage2_loss_analysis.png'
    plt.savefig(s2_plots_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n  Saved: {s2_plots_path}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 12: Stage 1 vs Stage 2 vs Baseline Comparison
# ============================================================
import json
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("  FINAL COMPARISON: Stage 1 vs Stage 2 vs Baseline")
print("=" * 70)

# Load results
s1_path = EVALS_DIR / "tl_stage1_results.json"
s2_path = EVALS_DIR / "tl_stage2_results.json"

s1 = json.load(open(s1_path)) if s1_path.exists() else None
s2 = json.load(open(s2_path)) if s2_path.exists() else None

# Baseline reference
BASELINE = {
    "mAP50": 0.788,
    "mAP50_95": 0.452,
    "precision": 0.719,
    "recall": 0.763,
}

if s1 and s2:
    # ── Comparison Table ──
    print(f"\n  {'Metric':<20} {'Baseline':>12} {'Stage 1':>12} {'Stage 2':>12}")
    print("  " + "-" * 60)

    for label, key in [('mAP@0.5', 'mAP50'), ('mAP@0.5:0.95', 'mAP50_95'),
                       ('Precision', 'precision'), ('Recall', 'recall')]:
        b = BASELINE[key] * 100
        v1 = s1[key] * 100
        v2 = s2[key] * 100
        print(f"  {label:<20} {b:>11.1f}% {v1:>11.1f}% {v2:>11.1f}%")

    # ── Stage 1 vs Stage 2 delta ──
    print(f"\n  {'Metric':<20} {'Stage 1':>12} {'Stage 2':>12} {'Delta':>12}")
    print("  " + "-" * 60)
    for label, key in [('mAP@0.5', 'mAP50'), ('mAP@0.5:0.95', 'mAP50_95'),
                       ('Precision', 'precision'), ('Recall', 'recall')]:
        v1 = s1[key] * 100
        v2 = s2[key] * 100
        d = v2 - v1
        sign = '+' if d > 0 else ''
        print(f"  {label:<20} {v1:>11.1f}% {v2:>11.1f}% {sign}{d:>11.1f}%")

    # ── Per-class comparison ──
    print(f"\n  Per-class AP@0.5:")
    print(f"  {'Class':<20} {'Stage 1':>12} {'Stage 2':>12} {'Delta':>12}")
    print("  " + "-" * 60)
    for cname in sorted(s1['per_class'].keys()):
        v1 = s1['per_class'][cname] * 100
        v2 = s2['per_class'][cname] * 100
        d = v2 - v1
        sign = '+' if d > 0 else ''
        print(f"  {cname:<20} {v1:>11.1f}% {v2:>11.1f}% {sign}{d:>11.1f}%")

    # ── Visual comparison ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Overall metrics
    labels = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall']
    baseline_vals = [BASELINE['mAP50']*100, BASELINE['mAP50_95']*100,
                     BASELINE['precision']*100, BASELINE['recall']*100]
    s1_vals = [s1['mAP50']*100, s1['mAP50_95']*100, s1['precision']*100, s1['recall']*100]
    s2_vals = [s2['mAP50']*100, s2['mAP50_95']*100, s2['precision']*100, s2['recall']*100]

    x = np.arange(len(labels))
    w = 0.25

    axes[0].bar(x - w, baseline_vals, w, label='Baseline (78.8%)', color='#95a5a6')
    axes[0].bar(x, s1_vals, w, label='Stage 1 (Feature Extraction)', color='#3498db')
    axes[0].bar(x + w, s2_vals, w, label='Stage 2 (Fine-Tuning)', color='#e74c3c')
    axes[0].set_ylabel('Score (%)')
    axes[0].set_title('Overall Metrics Comparison', fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    # Per-class
    classes = sorted(s1['per_class'].keys())
    s1_c = [s1['per_class'][c]*100 for c in classes]
    s2_c = [s2['per_class'][c]*100 for c in classes]

    x2 = np.arange(len(classes))
    axes[1].bar(x2 - w/2, s1_c, w, label='Stage 1', color='#3498db')
    axes[1].bar(x2 + w/2, s2_c, w, label='Stage 2', color='#e74c3c')
    axes[1].set_ylabel('AP@0.5 (%)')
    axes[1].set_title('Per-Class Comparison', fontweight='bold')
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels(classes, rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    comp_path = EVALS_DIR / 'tl_comparison.png'
    plt.savefig(comp_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n  Saved: {comp_path}")

    # ── Save final comparison ──
    final = {
        "baseline": BASELINE,
        "stage1": s1,
        "stage2": s2,
        "stage1_vs_baseline": {
            "mAP50_delta": s1['mAP50'] - BASELINE['mAP50'],
            "mAP50_95_delta": s1['mAP50_95'] - BASELINE['mAP50_95'],
        },
        "stage2_vs_baseline": {
            "mAP50_delta": s2['mAP50'] - BASELINE['mAP50'],
            "mAP50_95_delta": s2['mAP50_95'] - BASELINE['mAP50_95'],
        },
        "stage2_vs_stage1": {
            "mAP50_delta": s2['mAP50'] - s1['mAP50'],
            "mAP50_95_delta": s2['mAP50_95'] - s1['mAP50_95'],
        },
    }
    with open(EVALS_DIR / "tl_final_comparison.json", "w") as f:
        json.dump(final, f, indent=2)
    print(f"  Saved: {EVALS_DIR / 'tl_final_comparison.json'}")

else:
    print("\n  Results not found. Run Cell 6 and Cell 10 first.")

print("\n" + "=" * 70)

---

## Summary

### What This Notebook Does
| Stage | Frozen | Trainable | LR | Goal |
|-------|--------|-----------|-----|------|
| 1: Feature Extraction | 0-22 (all) | 23 (head) | 0.001 | Learn defect class mapping |
| 2: Fine-Tuning | 0-7 (early) | 8-23 (late+neck+head) | 0.0001 | Adapt to steel-specific features |

### Expected Results
| Stage | Expected mAP@0.5 | Notes |
|-------|------------------|-------|
| Baseline | 78.8% | Reference (600 epochs, no freezing) |
| Stage 1 | ~72-75% | Head-only, limited capacity |
| Stage 2 | ~78-82% | Neck adaptation improves detection |

### Files Generated
```
runs/detect/
├── tl_stage1_feature_extraction/weights/best.pt
└── tl_stage2_fine_tuning/weights/best.pt

evals/
├── tl_stage1_results.json
├── tl_stage2_results.json
├── tl_stage1_loss_analysis.png
├── tl_stage2_loss_analysis.png
├── tl_comparison.png
└── tl_final_comparison.json
```